# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule:** We will calculate the median CTR for every *position_tier* to establish a dynamic baseline. A page's *opportunity gap* is its tier's baseline CTR minus the actual CTR. The final *baseline_action_score* multiplies this gap by the log of its 90-day-impressions (to prioritize high-visibility pages). We will filter out the low-volume noise (*impressions_90d < 500*)

**Reason Codes**
- severe_ctr_underperformance: Gap is highly positive and impressions are very high.
- moderate_ctr_underperformance: Gap is positive with decent visibility. 
- performing_at_or_above_tier: CTR is equal to or better than the tier baseline.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Filter for visible pages to remove low-volume noise
df_visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)].copy()

# Calculate the dynamic baseline
tier_baselines = df_visible.groupby('position_tier')['ctr'].median().reset_index()
tier_baselines.rename(columns={'ctr': 'expected_tier_ctr'}, inplace=True)

# Merge back to main df
df_scored = pd.merge(df_visible, tier_baselines, on='position_tier', how='left')

# Calculate opportunity gap and score
df_scored['ctr_opportunity_gap'] = df_scored['expected_tier_ctr'] - df_scored['ctr']

# Score = Gap * log(impressions) to weight heavily toward high traffic pages
df_scored['baseline_action_score'] = np.where(
    df_scored['ctr_opportunity_gap'] > 0,
    df_scored['ctr_opportunity_gap'] * np.log1p(df_scored['impressions_90d']),
    0
)

# Assign reason codes
def assign_codes(row): 
    if row['baseline_action_score'] == 0:
        return 'performing_at_or_above_tier'
    elif row['ctr_opportunity_gap'] > 0.10 and row['impressions_90d'] > 5000:
        return 'severe_ctr_underperformance'
    else:
        return 'moderate_ctr_underperformance'

df_scored['reason_code'] = df_scored.apply(assign_codes, axis=1)

# Sort to create a ranked queue
df_ranked = df_scored.sort_values(by='baseline_action_score', ascending=False)

# Write to csv
os.makedirs('../../outputs', exist_ok=True)
df_ranked.to_csv('../../outputs/baseline_action_score.csv', index=False)
print(f"Ranked queue saved. Top score: {df_ranked['baseline_action_score'].max():.4f}")


Ranked queue saved. Top score: 2.9397


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Action for Top 20:** A human specialist should review the SERP intent for these URLs and rewrite the Title tag and Meta description for better match user expectations.  
**Reason Code:** Primarily *severe_ctr_underperformance* due to high impressions combined with CTRs vastly below their position tier average.   
**Confidence Note:** High confidence in the math, but moderate confidence in the business outcome.  
**What would make it wrong:** If the page ranks for highly informational, "zero-click" queries will naturally result in a  low CTR. Updating the metadata would not fix a SERP feature stealing clicks.

In [8]:
# Displaying the Top 20 for review
cols_to_review = [
    'content_id', 'impressions_90d', 'avg_position', 'position_tier', 
    'ctr', 'expected_tier_ctr', 'baseline_action_score', 'reason_code'
]
display(df_ranked[cols_to_review].head(20))

,content_id,impressions_90d,avg_position,position_tier,ctr,expected_tier_ctr,baseline_action_score,reason_code
4145,content_c8e9d6ab9013,208678,9.7,page_1,0.00,0.24,2.939653,severe_ctr_underperformance
15175,content_453722754fea,140079,7.6,page_1,0.01,0.24,2.725493,severe_ctr_underperformance
268,content_39881853ef0c,112434,7.2,page_1,0.01,0.24,2.674930,severe_ctr_underperformance
3831,content_c84a0ab98e90,223271,7.8,page_1,0.03,0.24,2.586391,severe_ctr_underperformance
8944,content_0919dd345d80,119217,7.0,page_1,0.02,0.24,2.571516,severe_ctr_underperformance
7640,content_d274ac4158ef,65138,6.8,page_1,0.01,0.24,2.549384,severe_ctr_underperformance
13879,content_e5f459e737b7,56363,5.9,page_1,0.01,0.24,2.516105,severe_ctr_underperformance
5122,content_c1fe78bc4e37,134055,7.5,page_1,0.03,0.24,2.479263,severe_ctr_underperformance
2517,content_339b357d04c7,46879,3.7,page_1,0.01,0.24,2.473730,severe_ctr_underperformance
1302,content_65114d89496d,72631,6.5,page_1,0.02,0.24,2.462495,severe_ctr_underperformance


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks:** If we look at ranks 50-100, we might see pages with exactly 501 impressions sneaking in. Because we used a hard threshold of 500, a page with 501 impressions and 0 clicks gets a high gap multiplier, even though it might be noise.  
**Leakage Check:** Confirmed that *clicks_90d* was completely excluded from the feature math to prevent target leakage, and no Flyrank product flags (*health_score, priority_score*) were used to build the queue. The baseline relies purely on observable search signals (*impressions, position, and CTR*).

## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.